In [3]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import HTML, display

ROOT = Path.cwd().parent
JSON_DIR = ROOT / "JSON_Edit"

json_paths = sorted(JSON_DIR.glob("*.json"))
json_files = [p.name for p in json_paths]

print(f"JSON folder: {JSON_DIR}")
print(f"Total JSON files found: {len(json_files)}")

if not json_files:
    raise FileNotFoundError(f"No JSON files found in {JSON_DIR}")

# Show clickable file links in the notebook.
links_html = "<br>".join(
    f"<a href='file:///{path.as_posix()}' target='_blank'>{path.name}</a>"
    for path in json_paths
)
display(HTML(f"<b>Detected JSON files (JSON_Edit):</b><br>{links_html}"))

print("\n" + "=" * 80)
print("CATEGORY SUMMARY BY JSON FILE")
print("=" * 80)

file_category_counts = {}
file_category_item_counts = {}
summary_rows = []

for path in json_paths:
    print(f"\nJSON File: {path.name}")

    with open(path, "r", encoding="utf-8") as f:
        items = json.load(f)

    categories = []
    for item in items:
        for prop in item.get("Properties", []):
            category = str(prop.get("category", "")).strip()
            if category:
                categories.append(category)

    if categories:
        counts = pd.Series(categories, name="Category").value_counts(dropna=False)
        file_category_counts[path.name] = counts

        per_file_df = (
            counts.rename_axis("Category")
            .reset_index(name="Count")
            .sort_values(["Category"], kind="stable")
            .reset_index(drop=True)
        )

        file_category_item_counts[path.name] = int(per_file_df["Count"].sum())

        summary_rows.append(
            {
                "JSON File": path.name,
                "Distinct Categories": int(per_file_df["Category"].nunique()),
                "Total Category Entries": int(per_file_df["Count"].sum()),
            }
        )
    else:
        per_file_df = pd.DataFrame([
            {"Category": "<no categories found>", "Count": 0}
        ])
        file_category_counts[path.name] = pd.Series(dtype="int64")
        file_category_item_counts[path.name] = 0

        summary_rows.append(
            {
                "JSON File": path.name,
                "Distinct Categories": 0,
                "Total Category Entries": 0,
            }
        )

    display(per_file_df)

print("\n" + "=" * 80)
print("JSON-LEVEL SUMMARY TABLE")
print("=" * 80)
json_summary_df = pd.DataFrame(summary_rows).sort_values(["JSON File"]).reset_index(drop=True)
display(json_summary_df)

print("\n" + "=" * 80)
print("ALL CATEGORIES ACROSS ALL JSON FILES")
print("=" * 80)

all_categories = sorted(
    {
        category
        for counts in file_category_counts.values()
        for category in counts.index.tolist()
    }
)

# Matrix table: category x file (counts)
matrix_rows = []
for category in all_categories:
    row = {"Category": category}
    for file_name in json_files:
        counts = file_category_counts.get(file_name, pd.Series(dtype="int64"))
        value = counts.get(category)
        row[file_name] = int(value) if pd.notna(value) else 0
    matrix_rows.append(row)

category_matrix_df = pd.DataFrame(matrix_rows)
with pd.option_context("display.max_rows", len(category_matrix_df), "display.max_columns", None):
    display(category_matrix_df)

# General roll-up table for quick planning.
overall_rows = []
for category in all_categories:
    present_files = 0
    total_count = 0
    for file_name in json_files:
        counts = file_category_counts.get(file_name, pd.Series(dtype="int64"))
        value = counts.get(category)
        if pd.notna(value):
            present_files += 1
            total_count += int(value)

    overall_rows.append(
        {
            "Category": category,
            "Files Present": present_files,
            "Total Entries": total_count,
        }
    )

overall_category_df = (
    pd.DataFrame(overall_rows)
    .sort_values(["Files Present", "Total Entries", "Category"], ascending=[False, False, True], kind="stable")
    .reset_index(drop=True)
)

print("\nOverall category roll-up:")
with pd.option_context("display.max_rows", len(overall_category_df), "display.max_columns", None):
    display(overall_category_df)

print(f"\nTotal unique categories across JSON_Edit: {len(all_categories)}")

JSON folder: c:\Git\APS-IFC\JSON_Edit
Total JSON files found: 7



CATEGORY SUMMARY BY JSON FILE

JSON File: ACD-18040-ALL-ST-N2x3.json


,Category,Count
0,Adaptive Component,16
1,Analysis Results,74
2,Analytical Properties,7881
3,Calculation Rules(Type),364
4,Constraints,66793
...,...,...
61,Structural Section Geometry,3528
62,Supports(Type),649
63,Text,6
64,Top Rail(Type),1395



JSON File: ASTIDC-STAN-HE-EPD-MVB1-M-E-0002.json


,Category,Count
0,ABB4HVDCDesign,168
1,ABB4HVDCDesignRevision,300
2,ABB4ModelRevision Master,52
3,DB Part,52
4,DXF_DWG,4
5,IFC,192
6,IFCAPPLICATION,3
7,IFCORGANIZATION,1
8,IFCOWNERHISTORY,3
9,IFCPERSON,3


...


ALL CATEGORIES ACROSS ALL JSON FILES


,Category,ACD-18040-ALL-ST-N2x3.json,ASTIDC-STAN-HE-EPD-MVB1-M-E-0002.json,ASTIDC-STAN-HE-MPD-RHP1-M-M-0001.json,ASTIDC-STAN-HE-MPD-TXBP1-M-M-0001.json,Ifc2x3_Duplex_Architecture.json,Ifc4_SampleHouse.json,Snowdon+Towers+Sample+Structural2x3.json
0,ABB4HVDCDesign,0,168,40252,8404,0,0,0
1,ABB4HVDCDesignRevision,0,300,53818,17456,0,0,0
2,ABB4ModelRevision Master,0,52,3504,1512,0,0,0
3,Adaptive Component,16,0,0,0,0,0,0
4,Analysis Results,74,0,0,0,0,0,0
5,Analytical Properties,7881,0,0,0,0,72,1038
6,Calculation Rules(Type),364,0,0,0,0,0,0
7,Classification:Uniformat Classification,0,0,0,0,0,56,0
8,Classification:Uniformat:A1010100,0,0,0,0,0,0,1078
9,Classification:Uniformat:A1010110,0,0,0,0,0,0,126



Overall category roll-up:


,Category,Files Present,Total Entries
0,Item,7,418319
1,IFC,7,118552
2,IFCAPPLICATION,6,18
3,IFCOWNERHISTORY,6,15
4,IFCPERSON,6,12
5,IFCORGANIZATION,6,6
6,Pset_SlabCommon,4,10445
7,IFCMATERIALLAYERSETUSAGE,4,8880
8,Pset_WallCommon,4,7024
9,IFCMATERIALLAYER,4,1484



Total unique categories across JSON_Edit: 136
